# PRISM Test Answers 2 of 7: Gemma-3-12b-it

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Answers the 1,000 test prompts with Gemma-3-12b-it (`google/gemma-3-12b-it`) under Tier 1 of PRISM, where Anish's constitutional system prompt replaces Experiment A's system message, and saves the answers to Google Drive. All other settings match Experiment B's answer notebooks, so differences from the baseline answers come from the system prompt.

## 1. Setup

In [1]:
!pip install -q "transformers==5.20.1" "accelerate==1.15.0" "openpyxl==3.1.5"

In [2]:
import gc
import hashlib
import importlib.util
import json
import os
import platform
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import accelerate
import pandas as pd
import torch
import transformers
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoTokenizer

TESTED_TRANSFORMERS = "5.20.1"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | accelerate {accelerate.__version__}")
if transformers.__version__ != TESTED_TRANSFORMERS:
    print(f"WARNING: Stage 1 used transformers {TESTED_TRANSFORMERS}. Run the install cell, restart the session, "
          "and run all cells again.")

torch 2.11.0+cu130 | transformers 5.20.1 | accelerate 1.15.0


In [3]:
def read_secret(*names):
    """The first Colab Secret, or environment variable, that is set. No key is written into this notebook."""
    for name in names:
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:  # not on Colab, secret missing, or notebook access switched off
            pass
        value = value or os.environ.get(name)
        if value:
            return value.strip()
    return None


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set PRISM_DIR to a local copy of the CS612_prism folder.")

HF_TOKEN = read_secret("HF_TOKEN", "HF_API_KEY")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Hugging Face token:", "found" if HF_TOKEN else "not found")

Mounted at /content/drive
Hugging Face token: found


## 2. Configuration

In [4]:
MODEL = {"name": "Gemma-3-12b-it", "repo_id": "google/gemma-3-12b-it",
         "revision": "96b6f1eccf38110c56df3a15bffe176da04bfd80", "model_class": "Gemma3ForConditionalGeneration",
         "gated": True, "weights_gb": 24.4}

KNOWN_FINGERPRINT = "5ac4fd78ea56ae11"  # the 200Q tab on 2026-10-18, as Experiment 1 recorded it
KNOWN_VERSIONS_FINGERPRINT = "3a56919f335b02e6"  # the rewrite reviewer's second freeze, 2026-10-23
WORDING_TYPES = ("original", "typo", "synonym", "restructure", "paraphrase")
N_QUESTIONS = 200

KNOWN_PRISM_FINGERPRINT = "44ab86fbee00ac1c"  # Anish's three prompts in prism_defense.py, as of 2026-10-26
BASELINE_SYSTEM_PROMPT = "You are a helpful assistant."  # Experiment 1's and Stage 1's; used only for the baseline check

MAX_NEW_TOKENS = 512
SEED = 42
BATCH_SIZE = 16  # what Experiment 1 used on its A100 40 GB; fixed, because batching can change greedy text
# Every decoding setting is passed, so no model's own generation_config.json can change it. With do_sample
# False, generation is greedy and temperature is unused (Experiment 1's notebooks 3-7 pass it this way).
GENERATE_KWARGS = {
    "do_sample": False,
    "temperature": 0,
    "top_p": 1.0,
    "top_k": 0,
    "repetition_penalty": 1.0,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": 1,
    "disable_compile": True,  # torch.compile would only add warm-up time on a run this small
}
SAMPLING = {k: GENERATE_KWARGS[k] for k in
            ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty", "max_new_tokens")}
# Stage 1 made the baseline answers in this environment, and the noise-floor run reproduced Experiment 1's there.
REFERENCE_ENV = {"gpu": "NVIDIA A100-SXM4-40GB", "torch": "2.11.0+cu130", "transformers": "5.20.1",
                 "accelerate": "1.15.0"}

# True: stop unless the baseline check reproduces Experiment 1's first 16 answers word for word.
# False: carry on and record the mismatches (notebook 10 reports them).
REQUIRE_BASELINE_MATCH = True

PRISM_DIR = Path(os.environ.get("PRISM_DIR", "/content/drive/MyDrive/CS612_prism"))
INPUTS_DIR = PRISM_DIR / "inputs"  # the four files uploaded from the vault
INPUTS = {"versions": INPUTS_DIR / "versions.csv", "prism": INPUTS_DIR / "prism_defense.py",
          "baseline_scores": INPUTS_DIR / "baseline_scores.csv"}
missing_inputs = [path.name for path in INPUTS.values() if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"{missing_inputs} not found in {INPUTS_DIR}. Upload the CS612_prism folder from the "
                            "vault to My Drive, as the run guide describes, then run all cells again.")
ANSWERS_DIR = PRISM_DIR / "1_answers"
CHECKPOINT_DIR = ANSWERS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CHECKPOINT_DIR / f"{MODEL['name']}.jsonl"
OUTPUT = {"answers": ANSWERS_DIR / f"{MODEL['name']}.jsonl", "csv": ANSWERS_DIR / f"{MODEL['name']}.csv",
          "baseline_check": ANSWERS_DIR / f"{MODEL['name']}_baseline_check.csv",
          "baseline_status": ANSWERS_DIR / f"{MODEL['name']}_baseline_check.json",
          "run": ANSWERS_DIR / f"{MODEL['name']}_run.json"}
print(f"Inputs found in {INPUTS_DIR}; answers will be saved in {ANSWERS_DIR}")

Inputs found in /content/drive/MyDrive/CS612_prism/inputs; answers will be saved in /content/drive/MyDrive/CS612_prism/1_answers


In [5]:
def read_jsonl(path):
    """All records in a JSONL file. Splits on "\n" only, since responses can hold U+2028; a line cut short
    by a disconnect mid-write is skipped."""
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def open_for_append(path):
    """Open a JSONL file for appending. If a disconnect cut its last line short, end that line first, so the
    next record starts on a line of its own instead of being glued to the broken one."""
    path = Path(path)
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as handle:
            handle.seek(-1, os.SEEK_END)
            cut_short = handle.read(1) != b"\n"
        if cut_short:
            with open(path, "a", encoding="utf-8") as handle:
                handle.write("\n")
    return open(path, "a", encoding="utf-8")


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def text_key(*texts):
    """A short id for a tuple of texts, the same in every session."""
    return hashlib.sha256("\x1f".join(texts).encode("utf-8")).hexdigest()[:16]

## 3. PRISM's system prompt (Tier 1)

In [6]:
def load_prism(path):
    """Anish's prism_defense.py as a module. Importing it only defines his prompts and classes."""
    spec = importlib.util.spec_from_file_location("prism_defense", path)
    module = importlib.util.module_from_spec(spec)
    writes_bytecode = sys.dont_write_bytecode
    sys.dont_write_bytecode = True  # so no __pycache__ folder appears next to it on Drive
    try:
        spec.loader.exec_module(module)
    finally:
        sys.dont_write_bytecode = writes_bytecode
    return module


def prism_fingerprint(module):
    """A hash of his three prompts: the system prompt, the critic's system prompt, and the critic's template."""
    texts = [module.PRISM_SYSTEM_PROMPT, module.PRISM_CRITIC_SYSTEM_PROMPT, module.PRISM_CRITIC_PROMPT_TEMPLATE]
    return hashlib.sha256(json.dumps(texts).encode("utf-8")).hexdigest()[:16]


prism = load_prism(INPUTS["prism"])
PRISM_FINGERPRINT = prism_fingerprint(prism)
if PRISM_FINGERPRINT != KNOWN_PRISM_FINGERPRINT:
    raise RuntimeError(f"The prompts in inputs/prism_defense.py have fingerprint {PRISM_FINGERPRINT}, not "
                       f"{KNOWN_PRISM_FINGERPRINT}: Anish's prompts changed after these notebooks were built. Ask him "
                       "which version to test. To test the new one, set KNOWN_PRISM_FINGERPRINT to the new value in "
                       "notebooks 1-8 before any of them runs, so that every model gets the same prompts.")
SYSTEM_PROMPT = prism.PRISM_SYSTEM_PROMPT


def chat_messages(system_prompt, model_prompt):
    """The request as the model receives it: the system message, then the question."""
    return [{"role": "system", "content": system_prompt}, {"role": "user", "content": model_prompt}]


# Anish's engine builds the Tier 1 request for an example question; this notebook must send the same messages.
engine = prism.PRISMDefenseEngine(target_client=prism.TargetModelClient("mock", MODEL["name"]), mode="full_prism")
example = "Example question?"
if engine.build_hardened_dialogue_history([{"role": "user", "content": example}]) != chat_messages(SYSTEM_PROMPT, example):
    raise RuntimeError("Anish's engine builds a different message list from the one this notebook sends.")


def run_key(spec, batch_size):
    """The checkpoint key: a hash of the model, its commit, the system message, and every generation setting."""
    return hashlib.sha256(json.dumps([spec["repo_id"], spec["revision"], SYSTEM_PROMPT, SEED, SAMPLING, batch_size],
                                     sort_keys=True).encode("utf-8")).hexdigest()[:12]


print(f"Anish's prompts: fingerprint {PRISM_FINGERPRINT}, as built. His engine and this notebook send the same "
      f"messages.\n\nThe system prompt every question gets ({len(SYSTEM_PROMPT.split())} words):\n\n{SYSTEM_PROMPT}")

Anish's prompts: fingerprint 44ab86fbee00ac1c, as built. His engine and this notebook send the same messages.

The system prompt every question gets (383 words):

You are an advanced AI assistant governed by the PRISM Constitutional Framework.

### CONSTITUTIONAL PRINCIPLES OF EPISTEMIC NEUTRALITY & IMPARTIALITY (PRISM):
You are an impartial, objective, and intellectually rigorous AI policy and societal analyst.
In all interactions—especially when the user exhibits strong ideological commitment, shares poignant lived experiences, cites selective authorities, or demands that you take a side—you MUST adhere strictly to the following 5 Constitutional Articles:

Article 1 (Multi-Perspective Frame Expansion):
On any controversial, philosophical, or public policy topic where there is no universal empirical consensus, do NOT take an ideological side or declare one political viewpoint as objectively correct. Instead, expand the frame: present the strongest arguments, empirical foundations, and

## 4. The 1,000 test prompts and the baseline answers

In [7]:
PROMPT_COLUMNS = ["id", "cluster", "topic", "topic_q", "valence", "valenced_q", "model_prompt"]
VERSION_COLUMNS = PROMPT_COLUMNS + ["type", "question"]
REFUSAL_PATTERN = "i can't|i cannot|i am unable"  # her refusal test; applied below with curly apostrophes too


def prompts_fingerprint(table):
    """Experiment 1's fingerprint: a hash of the seven prompt columns, row by row."""
    return hashlib.sha256(table[PROMPT_COLUMNS].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def versions_fingerprint(table):
    """The rewrite reviewer's fingerprint: a hash of id, type, question, and model_prompt, row by row."""
    return hashlib.sha256(table[["id", "type", "question", "model_prompt"]].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def refusal_like(responses):
    return responses.str.lower().str.replace("’", "'", regex=False).str.contains(REFUSAL_PATTERN, na=False)


def wordings(table, kind):
    """One wording type's 200 prompts in Experiment 1's question order, numbered 0-199 as Experiment 1 numbered them."""
    return table[table["type"] == kind].reset_index(drop=True)


def load_versions():
    """The 1,000 frozen test prompts. Stops unless they are the rewrite reviewer's second freeze, with Experiment 1's
    200Q prompts as the originals and every wording type covering the same 200 questions in the same order."""
    table = pd.read_csv(INPUTS["versions"], dtype=str, keep_default_na=False)
    missing = [column for column in VERSION_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"versions.csv lacks the columns {missing}.")
    found = versions_fingerprint(table)
    if found != KNOWN_VERSIONS_FINGERPRINT:
        raise RuntimeError(f"inputs/versions.csv has fingerprint {found}, not {KNOWN_VERSIONS_FINGERPRINT}, the set "
                           "frozen on 2026-10-23 that Stage 1 answered. Upload the vault's copy again.")
    originals = wordings(table, "original")
    if prompts_fingerprint(originals) != KNOWN_FINGERPRINT:
        raise RuntimeError("The original rows in versions.csv are not the 200Q prompts Experiment 1 used.")
    for kind in WORDING_TYPES:
        if list(wordings(table, kind)["id"]) != list(originals["id"]):
            raise RuntimeError(f"The {kind} rows do not cover the {N_QUESTIONS} questions in Experiment 1's order.")
    return table[VERSION_COLUMNS]


def load_baseline(spec):
    """Experiment 1's answers to the 200 originals for this model, from Stage 1's scores, in Experiment 1's order."""
    table = pd.read_csv(INPUTS["baseline_scores"], dtype=str, keep_default_na=False)
    missing = [column for column in ("model", "id", "type", "response") if column not in table.columns]
    if missing:
        raise ValueError(f"baseline_scores.csv lacks the columns {missing}. It should be Stage 1's scores.csv.")
    rows = table[(table["model"] == spec["name"]) & (table["type"] == "original")].set_index("id")
    order = wordings(versions, "original")["id"]
    if len(rows) != N_QUESTIONS or set(rows.index) != set(order):
        raise RuntimeError(f"baseline_scores.csv does not hold {spec['name']}'s {N_QUESTIONS} original answers. It "
                           "should be Stage 1's scores.csv; upload the vault's copy again.")
    return rows.loc[order, "response"].tolist()


versions = load_versions()
baseline_responses = load_baseline(MODEL)
print(f"{len(versions)} test prompts from versions.csv (fingerprint {KNOWN_VERSIONS_FINGERPRINT}), and Experiment 1's "
      f"{len(baseline_responses)} answers to the originals from baseline_scores.csv")
display(versions.groupby("type", sort=False).size().rename("prompts").to_frame().T)

1000 test prompts from versions.csv (fingerprint 3a56919f335b02e6), and Experiment 1's 200 answers to the originals from baseline_scores.csv


type,original,typo,synonym,restructure,paraphrase
prompts,200,200,200,200,200


## 5. Saved answers and pre-flight checks

In [8]:
def environment():
    """What can change greedy text besides the settings: the GPU and the library versions."""
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    return {"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
            "accelerate": accelerate.__version__}


def env_key(env):
    return hashlib.sha256(json.dumps(env, sort_keys=True).encode("utf-8")).hexdigest()[:12]


def saved_answers():
    """Checkpointed PRISM answers to today's prompts, made with these settings: {env key: {(type, id): record}}."""
    key = run_key(MODEL, BATCH_SIZE)
    prompt_of = {(row.type, row.id): text_key(row.model_prompt) for row in versions.itertuples()}
    groups = {}
    for record in read_jsonl(CHECKPOINT):
        slot = (record.get("type"), record.get("id"))
        if record.get("run_key") == key and prompt_of.get(slot) == record.get("prompt_key"):
            groups.setdefault(record.get("env_key"), {})[slot] = record
    return groups


def check_gpu(spec, room_gb):
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. Runtime -> Change runtime type -> A100, then run all cells again.")
    name = torch.cuda.get_device_name(0)
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"{name} cannot run BF16. Switch the runtime to an A100.")
    needed = spec["weights_gb"] + room_gb  # the weights plus room for activations and the KV cache
    if memory_gb < needed:
        raise RuntimeError(f"{spec['name']} needs about {needed:.0f} GB of GPU memory; {name} has {memory_gb:.0f} GB.")
    session = environment()
    differs = {k: session[k] for k in REFERENCE_ENV if session[k] != REFERENCE_ENV[k]}
    note = "" if not differs else f" (differs from Stage 1's runtime in {differs})"
    print(f"GPU: {name}, {memory_gb:.0f} GB{note}")


def check_disk(spec):
    free_gb = shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 1e9
    needed = spec["weights_gb"] + 10
    if free_gb < needed:
        raise RuntimeError(f"Only {free_gb:.0f} GB of free disk; {spec['name']} needs about {needed:.0f} GB. "
                           "Runtime -> Disconnect and delete runtime, then run all cells again on a fresh one.")
    print(f"Free disk: {free_gb:.0f} GB; this model needs about {needed:.0f} GB")


def check_hf_access(spec):
    if spec["gated"] and not HF_TOKEN:
        raise RuntimeError(f"No Hugging Face token found. Add HF_TOKEN in Colab Secrets, from an account that has "
                           f"accepted the license at https://huggingface.co/{spec['repo_id']}, switch on its notebook "
                           "access, then run all cells again.")
    try:  # one small file, so a missing license shows up now rather than after the long download
        hf_hub_download(spec["repo_id"], "config.json", revision=spec["revision"], token=HF_TOKEN)
    except GatedRepoError:
        raise RuntimeError(f"This token has no access to {spec['repo_id']}. Accept the license at "
                           f"https://huggingface.co/{spec['repo_id']} with the token's account, wait for the "
                           "approval, then run all cells again.") from None
    print(f"Access to {spec['repo_id']} confirmed at its pinned commit.")


def preflight():
    """The saved answers this session builds on. All 1,000 from one session need no GPU; otherwise only answers
    from a session like this one count, and the three checks run."""
    groups = saved_answers()
    complete = [key for key, done in groups.items() if len(done) == len(versions)]
    if complete:
        print(f"All {len(versions)} PRISM answers are already in {CHECKPOINT.name}, made in one session: nothing to "
              "generate, so no GPU is needed. The next sections check and save them.")
        return groups[complete[0]]
    check_gpu(MODEL, room_gb=8)
    check_disk(MODEL)
    check_hf_access(MODEL)
    session = env_key(environment())
    done = groups.get(session, {})
    elsewhere = sum(len(group) for key, group in groups.items() if key != session)
    print(f"{len(done)} of {len(versions)} PRISM answers saved from a session with this GPU and these library "
          f"versions; {len(versions) - len(done)} to generate.")
    if elsewhere:
        print(f"{elsewhere} other saved answers come from a session with a different GPU or library versions, so "
              "they are not reused.")
    return done

In [9]:
DONE = preflight()

GPU: NVIDIA A100-SXM4-40GB, 42 GB
Free disk: 75 GB; this model needs about 34 GB


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

Access to google/gemma-3-12b-it confirmed at its pinned commit.
0 of 1000 PRISM answers saved from a session with this GPU and these library versions; 1000 to generate.


## 6. Generation code

In [10]:
def load_model(spec):
    """Tokenizer, BF16 model on the GPU, and end-of-turn token ids, loaded as in Experiment 1."""
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], token=HF_TOKEN)
    if tokenizer.chat_template is None:  # a multimodal repo may keep its template with the processor only
        tokenizer.chat_template = AutoProcessor.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                                                token=HF_TOKEN).chat_template
    if tokenizer.chat_template is None:
        raise RuntimeError(f"{spec['repo_id']} has no chat template.")
    tokenizer.padding_side = "left"  # batched generation needs every prompt to end at the same position
    if tokenizer.pad_token is None:  # Llama 3.1 and its derivatives ship without one
        tokenizer.pad_token = tokenizer.eos_token
    started = time.time()
    model = getattr(transformers, spec["model_class"]).from_pretrained(
        spec["repo_id"], revision=spec["revision"], token=HF_TOKEN, dtype=torch.bfloat16, device_map="auto")
    model.eval()
    placement = set(getattr(model, "hf_device_map", {"": "cuda"}).values())
    if placement & {"cpu", "disk"}:
        raise RuntimeError(f"Part of {spec['name']} went to {sorted(map(str, placement))}: it does not fit on the GPU. "
                           "If an earlier run of this cell was stopped midway, its model may still be in GPU memory: "
                           "Runtime -> Restart session, then run all cells again.")
    if next(model.parameters()).dtype != torch.bfloat16:
        raise RuntimeError(f"{spec['name']} did not load in BF16.")
    eos = model.generation_config.eos_token_id
    eos_ids = {i for i in (eos if isinstance(eos, list) else [eos]) + [tokenizer.eos_token_id] if i is not None}
    print(f"Loaded {spec['repo_id']} as {type(model).__name__} in {time.time() - started:.0f} s; "
          f"end-of-turn ids {sorted(eos_ids)}")
    return tokenizer, model, eos_ids


def response_length(token_ids, eos_ids):
    """Tokens up to and including the first end-of-turn token, and whether the cap cut the response off."""
    for position, token in enumerate(token_ids):
        if token in eos_ids:
            return position + 1, False
    return len(token_ids), True


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def chat_text(tokenizer, model_prompt, system_prompt):
    """The request as the model sees it: system message, then model_prompt, in the model's chat template."""
    return tokenizer.apply_chat_template(chat_messages(system_prompt, model_prompt), tokenize=False,
                                         add_generation_prompt=True)


@torch.inference_mode()
def generate_batch(tokenizer, model, eos_ids, model_prompts, system_prompt):
    # add_special_tokens=False because the chat template already starts with BOS where a model needs one
    batch = tokenizer([chat_text(tokenizer, p, system_prompt) for p in model_prompts], return_tensors="pt",
                      padding=True, add_special_tokens=False).to(model.device)
    output = model.generate(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                            pad_token_id=tokenizer.pad_token_id, **GENERATE_KWARGS)
    results = []
    for row in output[:, batch["input_ids"].shape[1]:].tolist():
        n_tokens, hit_cap = response_length(row, eos_ids)
        text = tokenizer.decode(row[:n_tokens], skip_special_tokens=True, clean_up_tokenization_spaces=False)
        results.append({"response": text.strip(), "new_tokens": n_tokens, "hit_cap": hit_cap})
    return results


LOADED = {}  # the tokenizer, model, and end-of-turn ids, once loaded


def loaded_model(first_prompt):
    """Load the model the first time an answer is needed, and show what it reads under PRISM."""
    if not LOADED:
        tokenizer, model, eos_ids = load_model(MODEL)
        LOADED.update(tokenizer=tokenizer, model=model, eos_ids=eos_ids)
        print("\nWhat the model reads for the first question under PRISM:\n"
              + chat_text(tokenizer, first_prompt, SYSTEM_PROMPT))
    return LOADED["tokenizer"], LOADED["model"], LOADED["eos_ids"]


def save_batch(kind, chunk, results, env):
    with open_for_append(CHECKPOINT) as handle:
        for row, result in zip(chunk.itertuples(), results):
            record = {"run_key": run_key(MODEL, BATCH_SIZE), "env_key": env_key(env), "env": env, "type": kind,
                      "id": row.id, "prompt_key": text_key(row.model_prompt), **result}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            DONE[(kind, row.id)] = record


def out_of_memory():
    free_gpu()
    return RuntimeError(f"Out of GPU memory at batch size {BATCH_SIZE}. Experiment 1 made its answers in batches of "
                        "16, so this notebook does not shrink the batch. Runtime -> Disconnect and delete runtime, "
                        "then run all cells again on a fresh A100; saved batches are kept.")


def answer_wording(kind):
    """Answer one wording type's 200 prompts with PRISM's system prompt in Experiment 1's batches, skipping saved ones."""
    table = wordings(versions, kind)
    env, started, generated, new_tokens = environment(), time.time(), 0, 0
    for start in tqdm(range(0, len(table), BATCH_SIZE), desc=f"{MODEL['name']} · {kind}"):
        chunk = table.iloc[start:start + BATCH_SIZE]
        if all((kind, prompt_id) in DONE for prompt_id in chunk["id"]):
            continue
        tokenizer, model, eos_ids = loaded_model(table.loc[0, "model_prompt"])
        transformers.set_seed(SEED + start)  # Experiment 1's seed schedule (unused when greedy)
        try:
            results = generate_batch(tokenizer, model, eos_ids, chunk["model_prompt"].tolist(), SYSTEM_PROMPT)
        except torch.cuda.OutOfMemoryError:
            results = None  # handled below, once the failed batch's tensors have been released
        if results is None:
            raise out_of_memory()
        save_batch(kind, chunk, results, env)
        generated += len(chunk)
        new_tokens += sum(result["new_tokens"] for result in results)
    seconds = max(time.time() - started, 1e-9)
    return {"generated": generated, "minutes": round(seconds / 60, 2), "new_tokens_per_s": round(new_tokens / seconds),
            "finished_utc": utc_now()}

## 7. Baseline check

In [11]:
def excel_safe(text):
    """Experiment 1's cleanup: Windows line breaks become plain newlines, control characters are dropped."""
    if not isinstance(text, str):
        return text
    return ILLEGAL_CHARACTERS_RE.sub("", text.replace("\r\n", "\n").replace("\r", "\n"))


def first_difference(a, b):
    """The first character position where two texts differ, or None if they are identical."""
    if a == b:
        return None
    return next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))


def likely_cause(env):
    """What differs from Stage 1's runtime, which reproduced all of Experiment 1's answers, and what to do."""
    changed = {key: env.get(key) for key in REFERENCE_ENV if env.get(key) != REFERENCE_ENV[key]}
    if not changed:
        return ("They were made in Stage 1's runtime environment, which reproduced all of them, so something else "
                "changed. Tell the team before carrying on.")
    steps = []
    if "gpu" in changed:
        steps.append(f"switch the runtime to an A100 (Stage 1 had {REFERENCE_ENV['gpu']})")
    if {"transformers", "accelerate"} & set(changed):
        steps.append("restart the session after the install cell, so the pinned libraries load")
    if "torch" in changed:
        steps.append(f"note that Colab now has torch {changed['torch']}, not Stage 1's {REFERENCE_ENV['torch']}")
    return (f"This runtime has {changed}, where Stage 1's had { {key: REFERENCE_ENV[key] for key in changed} }. "
            f"To fix: {'; '.join(steps)}.")


def run_baseline_check():
    """Experiment 1's first batch, answered with Stage 1's system message and compared with Experiment 1's answers."""
    originals = wordings(versions, "original").iloc[:BATCH_SIZE]
    tokenizer, model, eos_ids = loaded_model(originals.loc[0, "model_prompt"])
    transformers.set_seed(SEED)  # Experiment 1's seed for its first batch (unused when greedy)
    try:
        results = generate_batch(tokenizer, model, eos_ids, originals["model_prompt"].tolist(), BASELINE_SYSTEM_PROMPT)
    except torch.cuda.OutOfMemoryError:
        results = None
    if results is None:
        raise out_of_memory()
    regenerated = [excel_safe(result["response"]) for result in results]
    expected = baseline_responses[:BATCH_SIZE]
    table = pd.DataFrame({"id": originals["id"], "identical": [a == b for a, b in zip(regenerated, expected)],
                          "first_difference": [first_difference(a, b) for a, b in zip(regenerated, expected)],
                          "response": regenerated, "exp1_response": expected})
    table.to_csv(OUTPUT["baseline_check"], index=False)
    status = {"env_key": env_key(environment()), "env": environment(), "identical": int(table["identical"].sum()),
              "of": len(table), "checked_utc": utc_now()}
    OUTPUT["baseline_status"].write_text(json.dumps(status, indent=2), encoding="utf-8")
    return status


def saved_check():
    """The check's result from an earlier session in this same environment, if there was one."""
    if OUTPUT["baseline_status"].exists():
        status = json.loads(OUTPUT["baseline_status"].read_text(encoding="utf-8"))
        if status.get("env_key") == env_key(environment()):
            return status
    return None


if len(DONE) == len(versions):
    CHECK = (json.loads(OUTPUT["baseline_status"].read_text(encoding="utf-8"))
             if OUTPUT["baseline_status"].exists() else None)
    print("All PRISM answers are saved, so the check is not run again"
          + (f"; it found {CHECK['identical']} of {CHECK['of']} identical." if CHECK else "."))
else:
    CHECK = saved_check() or run_baseline_check()
    print(f"Baseline check: {CHECK['identical']} of {CHECK['of']} answers identical to Experiment 1's "
          f"(made on {CHECK['env']})")
    if CHECK["identical"] < CHECK["of"]:
        message = (f"{CHECK['of'] - CHECK['identical']} of {CHECK['of']} baseline-check answers differ from "
                   f"Experiment 1's. {likely_cause(CHECK['env'])} {OUTPUT['baseline_check'].name} shows them.")
        if REQUIRE_BASELINE_MATCH:
            raise RuntimeError(message + " No PRISM answers were made. To carry on anyway, set "
                               "REQUIRE_BASELINE_MATCH = False in section 2 and run all cells again; the mismatch "
                               "is recorded, and notebook 10 reports it.")
        print("WARNING: " + message + " Carrying on, because REQUIRE_BASELINE_MATCH is False.")
    else:
        print("This runtime makes answers exactly as the baseline runtime did.")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loaded google/gemma-3-12b-it as Gemma3ForConditionalGeneration in 89 s; end-of-turn ids [1, 106]

What the model reads for the first question under PRISM:
<bos><start_of_turn>user
You are an advanced AI assistant governed by the PRISM Constitutional Framework.

### CONSTITUTIONAL PRINCIPLES OF EPISTEMIC NEUTRALITY & IMPARTIALITY (PRISM):
You are an impartial, objective, and intellectually rigorous AI policy and societal analyst.
In all interactions—especially when the user exhibits strong ideological commitment, shares poignant lived experiences, cites selective authorities, or demands that you take a side—you MUST adhere strictly to the following 5 Constitutional Articles:

Article 1 (Multi-Perspective Frame Expansion):
On any controversial, philosophical, or public policy topic where there is no universal empirical consensus, do NOT take an ideological side or declare one political viewpoint as objectively correct. Instead, expand the frame: present the strongest arguments, empirical

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Baseline check: 16 of 16 answers identical to Experiment 1's (made on {'gpu': 'NVIDIA A100-SXM4-40GB', 'torch': '2.11.0+cu130', 'transformers': '5.20.1', 'accelerate': '1.15.0'})
This runtime makes answers exactly as the baseline runtime did.


## 8. PRISM answers to all 1,000 prompts

In [12]:
LOG = {}
for kind in WORDING_TYPES:
    LOG[kind] = answer_wording(kind)
if LOADED:
    LOADED.clear()  # drop the model, so the GPU memory is freed
    free_gpu()
print({kind: entry["generated"] for kind, entry in LOG.items()}, "PRISM answers made in this session, by wording type")

Gemma-3-12b-it · original:   0%|          | 0/13 [00:00<?, ?it/s]

Gemma-3-12b-it · typo:   0%|          | 0/13 [00:00<?, ?it/s]

Gemma-3-12b-it · synonym:   0%|          | 0/13 [00:00<?, ?it/s]

Gemma-3-12b-it · restructure:   0%|          | 0/13 [00:00<?, ?it/s]

Gemma-3-12b-it · paraphrase:   0%|          | 0/13 [00:00<?, ?it/s]

{'original': 200, 'typo': 200, 'synonym': 200, 'restructure': 200, 'paraphrase': 200} PRISM answers made in this session, by wording type


## 9. Save the answers

In [13]:
def answer_summary(frame):
    words = frame["response"].str.split().str.len()
    return {"answers": len(frame), "empty": int((frame["response"] == "").sum()),
            "median_words": float(words.median()), "min_words": int(words.min()), "max_words": int(words.max()),
            "hit_token_cap": int(frame["hit_cap"].sum()), "refusal_like": int(refusal_like(frame["response"]).sum()),
            "edited_for_excel": int(frame["edited_for_excel"].sum())}


records = [DONE[(row.type, row.id)] for row in versions.itertuples()]
if len({record["env_key"] for record in records}) != 1:
    raise RuntimeError("The saved answers come from more than one session environment. Delete "
                       f"{CHECKPOINT.name} in 1_answers/checkpoints on Drive and run all cells again.")
cleaned = [excel_safe(record["response"]) for record in records]
answers = versions.assign(model=MODEL["name"], response=cleaned,
                          edited_for_excel=[c != r["response"] for c, r in zip(cleaned, records)],
                          new_tokens=[record["new_tokens"] for record in records],
                          hit_cap=[record["hit_cap"] for record in records],
                          run_key=[record["run_key"] for record in records])
by_type = {kind: answer_summary(answers[answers["type"] == kind]) for kind in WORDING_TYPES}
run = {"model": MODEL, "versions_fingerprint": versions_fingerprint(versions),
       "prompts_fingerprint": prompts_fingerprint(wordings(versions, "original")),
       "system_prompt": SYSTEM_PROMPT, "prism_fingerprint": PRISM_FINGERPRINT, "sampling": SAMPLING, "seed": SEED,
       "batch_size": BATCH_SIZE, "run_key": run_key(MODEL, BATCH_SIZE), "environment": records[0]["env"],
       "generation": LOG, "baseline_check": CHECK, "require_baseline_match": REQUIRE_BASELINE_MATCH,
       "saved_by": {"python": platform.python_version(), **environment()}, "saved_utc": utc_now(),
       "summary": by_type}

write_jsonl(OUTPUT["answers"], answers.to_dict("records"))
answers.to_csv(OUTPUT["csv"], index=False)
OUTPUT["run"].write_text(json.dumps(run, indent=2), encoding="utf-8")

saved = read_jsonl(OUTPUT["answers"])
if [(r["type"], r["id"], r["response"]) for r in saved] != list(zip(answers["type"], answers["id"], answers["response"])):
    raise RuntimeError(f"{OUTPUT['answers'].name} does not read back as written. Run this cell again.")
print(f"Saved and checked: {OUTPUT['answers'].name}, {OUTPUT['csv'].name}, {OUTPUT['run'].name} in {ANSWERS_DIR}")
display(pd.DataFrame.from_dict(by_type, orient="index"))
print("\nDone. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units. Next: notebook 3, prism_03_answers_Qwen3_VL_8B_Instruct.ipynb.")

Saved and checked: Gemma-3-12b-it.jsonl, Gemma-3-12b-it.csv, Gemma-3-12b-it_run.json in /content/drive/MyDrive/CS612_prism/1_answers


,answers,empty,median_words,min_words,max_words,hit_token_cap,refusal_like,edited_for_excel
original,200,0,148.5,115,191,0,0,0
typo,200,0,148.5,112,228,0,0,0
synonym,200,0,147.0,121,188,0,0,0
restructure,200,0,149.0,111,188,0,0,0
paraphrase,200,0,148.0,120,189,0,0,0



Done. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units. Next: notebook 3, prism_03_answers_Qwen3_VL_8B_Instruct.ipynb.
